# Stage1 GradCAM Viewer

`train_models/stage1/gradcam_viz.py` の GradCAM 可視化を notebook から実行するための版です。

- 先頭の Parameters セルだけで `FOLD`, `CATEGORY`, `N`, `GPU`, `DISPLAY_MODE` を設定します。
- `DISPLAY_MODE="all"` は全スライス、`"orientation"` は向き補正使用スライス、`"bbox"` は bbox ありスライスだけを表示します。
- bbox は赤枠、vertebra mask は緑輪郭で重ねます。
- `show_sample(index)` で任意のサンプルを表示します。
- `ipywidgets` が使える環境では、最後のセルでスライダー操作できます。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "train_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"NOTEBOOK_DIR={NOTEBOOK_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

from gradcam_viz import (
    DATASET_DIR,
    build_figure,
    build_oof,
    compute_gradcam,
    get_bbox_forced,
    get_bbox_overlays,
    get_orientation_forced,
    load_model,
)

plt.rcParams["figure.facecolor"] = "white"

## Parameters

In [ ]:
FOLD = 0
CATEGORY = "TP"  # one of: FN, FP, TP, TN
N = 50
GPU = 1
START_IDX = 0

# one of: "all", "orientation", "bbox"
DISPLAY_MODE = "bbox"

# Backward-compatible flags. Usually do not edit these directly.
ORIENTATION_ONLY = DISPLAY_MODE == "orientation"
BBOX_ONLY = DISPLAY_MODE == "bbox"

SAVE_DPI = 120

In [ ]:
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

oof_df = build_oof(FOLD)

ascending = CATEGORY in ("FN", "TN")
candidate_df = (
    oof_df[(oof_df["category"] == CATEGORY) & (oof_df["fold"] == FOLD)]
    .sort_values("pred_prob", ascending=ascending)
    .reset_index(drop=True)
)

if DISPLAY_MODE == "bbox":
    has_target = candidate_df.apply(
        lambda row: bool(get_bbox_forced(row["study_uid"], row["vertebra"])),
        axis=1,
    )
    candidate_df = candidate_df[has_target]
elif DISPLAY_MODE == "orientation":
    has_target = candidate_df.apply(
        lambda row: bool(get_orientation_forced(row["study_uid"], row["vertebra"])),
        axis=1,
    )
    candidate_df = candidate_df[has_target]
elif DISPLAY_MODE != "all":
    raise ValueError(f"DISPLAY_MODE must be one of 'all', 'orientation', 'bbox', got {DISPLAY_MODE!r}")

filtered = candidate_df.head(N).reset_index(drop=True)
samples = filtered.to_dict("records")

print(f"{CATEGORY} x fold{FOLD} x mode={DISPLAY_MODE}: {len(filtered)} / {len(candidate_df)} 件")
display(filtered[["study_uid", "vertebra", "label", "pred_prob", "category", "fold"]].head(20))

In [ ]:
model = load_model(FOLD, device)

## Single-sample display

`show_sample(0)` のように index を指定して表示します。通常は Parameters セルの `DISPLAY_MODE` を変えるだけで表示対象を切り替えます。

In [ ]:
def selected_display_indices(
    bbox_forced: set[int],
    orientation_forced: set[int],
    display_mode: str | None = None,
) -> list[int] | None:
    mode = DISPLAY_MODE if display_mode is None else display_mode
    if mode == "bbox":
        return sorted(bbox_forced)
    if mode == "orientation":
        return sorted(orientation_forced)
    if mode == "all":
        return None
    raise ValueError(f"DISPLAY_MODE must be one of 'all', 'orientation', 'bbox', got {mode!r}")


def show_sample(
    index: int,
    save: bool = False,
    display_mode: str | None = None,
    orientation_only: bool | None = None,
    bbox_only: bool | None = None,
) -> plt.Figure:
    if not samples:
        raise ValueError("No samples. Change FOLD, CATEGORY, or N.")
    if index < 0 or index >= len(samples):
        raise IndexError(f"index must be in [0, {len(samples) - 1}], got {index}")

    mode = DISPLAY_MODE if display_mode is None else display_mode
    if bbox_only is True:
        mode = "bbox"
    elif orientation_only is True:
        mode = "orientation"
    elif bbox_only is False and orientation_only is False and display_mode is None:
        mode = "all"

    row = samples[index]
    study_uid = row["study_uid"]
    vertebra = row["vertebra"]
    print(f"[{index + 1}/{len(samples)}] {study_uid} / {vertebra}  mode={mode}  (computing GradCAM...)")

    ct = np.load(DATASET_DIR / study_uid / vertebra / "ct.npy", allow_pickle=False)
    mask = np.load(DATASET_DIR / study_uid / vertebra / "vertebra_mask.npy", allow_pickle=False)
    bbox_forced = get_bbox_forced(study_uid, vertebra)
    orientation_forced = get_orientation_forced(study_uid, vertebra)
    bbox_overlays = get_bbox_overlays(study_uid, vertebra)
    display_indices = selected_display_indices(bbox_forced, orientation_forced, mode)

    if mode == "bbox":
        print(f"bbox slices: {display_indices}")
    elif mode == "orientation":
        print(f"orientation slices: {display_indices}")

    cams, probs = compute_gradcam(model, ct, mask, device)

    oof_row = oof_df[(oof_df["study_uid"] == study_uid) & (oof_df["vertebra"] == vertebra)]
    gt = int(oof_row["label"].values[0]) if len(oof_row) else -1
    pred_prob = float(oof_row["pred_prob"].values[0]) if len(oof_row) else float("nan")
    category = oof_row["category"].values[0] if len(oof_row) else "?"

    fig = build_figure(
        study_uid,
        vertebra,
        cams,
        probs,
        ct,
        mask,
        bbox_forced,
        orientation_forced,
        bbox_overlays,
        gt,
        pred_prob,
        category,
        index,
        len(samples),
        display_indices=display_indices,
    )

    if save:
        out = Path(f"gradcam_{study_uid}_{vertebra}.png")
        fig.savefig(out, dpi=SAVE_DPI, bbox_inches="tight", facecolor="#1a1a1a")
        print(f"saved: {out}")

    return fig

In [ ]:
fig = show_sample(START_IDX)

## Slider viewer

`ipywidgets` が入っている場合、サンプル index をスライダーまたは前後ボタンで切り替えて表示します。初期値は Parameters セルの `DISPLAY_MODE` を使います。

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    current_fig = {"fig": None}

    index_slider = widgets.IntSlider(
        value=min(START_IDX, max(len(samples) - 1, 0)),
        min=0,
        max=max(len(samples) - 1, 0),
        step=1,
        description="sample",
        continuous_update=False,
        layout=widgets.Layout(width="600px"),
    )
    prev_button = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="90px"))
    next_button = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))
    save_button = widgets.Button(description="Save current PNG", layout=widgets.Layout(width="150px"))
    mode_dropdown = widgets.Dropdown(
        options=[("all slices", "all"), ("orientation only", "orientation"), ("bbox only", "bbox")],
        value=DISPLAY_MODE,
        description="mode",
        layout=widgets.Layout(width="220px"),
    )
    status = widgets.HTML()
    output = widgets.Output()

    def render(change=None) -> None:
        with output:
            clear_output(wait=True)
            if current_fig["fig"] is not None:
                plt.close(current_fig["fig"])
            fig = show_sample(
                index_slider.value,
                save=False,
                display_mode=mode_dropdown.value,
            )
            current_fig["fig"] = fig
            status.value = f"<b>{index_slider.value + 1}</b> / {len(samples)}"
            display(fig)

    def go_prev(_button) -> None:
        index_slider.value = max(index_slider.min, index_slider.value - 1)

    def go_next(_button) -> None:
        index_slider.value = min(index_slider.max, index_slider.value + 1)

    def save_current(_button) -> None:
        with output:
            row = samples[index_slider.value]
            out = Path(f"gradcam_{row['study_uid']}_{row['vertebra']}.png")
            fig = current_fig["fig"]
            if fig is None:
                print("No figure to save yet.")
                return
            fig.savefig(out, dpi=SAVE_DPI, bbox_inches="tight", facecolor="#1a1a1a")
            print(f"saved: {out}")

    index_slider.observe(render, names="value")
    mode_dropdown.observe(render, names="value")
    prev_button.on_click(go_prev)
    next_button.on_click(go_next)
    save_button.on_click(save_current)

    controls = widgets.VBox([
        widgets.HBox([prev_button, next_button, index_slider, status]),
        widgets.HBox([mode_dropdown, save_button]),
    ])
    display(widgets.VBox([controls, output]))
    render()
except ImportError as error:
    raise ImportError(
        "Slider viewer requires ipywidgets. "
        "Install project dependencies with `uv sync`, then restart the notebook kernel."
    ) from error